In [8]:

from pathlib import Path
import pandas as pd
import numpy as np

SEASON = 2026
TARGET_WEEK = 7

PROJECT_ROOT = Path("../..").resolve()
WEEKLY_DATA_DIR = (
    PROJECT_ROOT / "data" / "processed" / "weekly"
)
OUTPUT_DIR = (
    PROJECT_ROOT / "rest_of_season" / "outputs"
    / f"week_{TARGET_WEEK:02d}"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("ROS output:", OUTPUT_DIR)


Project root: C:\Users\efriedman\Desktop\NFL-Season-Projections
ROS output: C:\Users\efriedman\Desktop\NFL-Season-Projections\rest_of_season\outputs\week_07


In [9]:

schedule_path = (
    WEEKLY_DATA_DIR / f"{SEASON}_schedule_current.parquet"
)
completed_path = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_completed_games.parquet"
)
strength_path = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_sequential_team_strength.parquet"
)
injury_path = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_injury_adjusted_team_strength.parquet"
)

required_files = [
    schedule_path,
    completed_path,
    strength_path,
    injury_path
]

for path in required_files:
    if not path.exists():
        raise FileNotFoundError(
            f"Required pipeline output missing: {path}"
        )

schedule = pd.read_parquet(schedule_path)
completed = pd.read_parquet(completed_path)
strength = pd.read_parquet(strength_path)
injury_strength = pd.read_parquet(injury_path)

print("Schedule:", schedule.shape)
print("Completed:", completed.shape)
print("Strength:", strength.shape)
print("Injury strength:", injury_strength.shape)


Schedule: (272, 46)
Completed: (93, 46)
Strength: (32, 25)
Injury strength: (32, 30)


In [10]:

remaining_games = schedule.loc[
    (schedule["week"] >= TARGET_WEEK)
    & schedule["home_score"].isna()
    & schedule["away_score"].isna()
].copy()

remaining_games = remaining_games.sort_values(
    ["week", "gameday", "game_id"]
).reset_index(drop=True)

assert remaining_games["game_id"].is_unique
assert remaining_games["home_team"].notna().all()
assert remaining_games["away_team"].notna().all()

print("Remaining regular-season games:", len(remaining_games))

display(
    remaining_games[
        ["week", "gameday", "away_team", "home_team"]
    ].head(20)
)


Remaining regular-season games: 179


,week,gameday,away_team,home_team
0,7,2026-10-22,NE,CHI
1,7,2026-10-25,CIN,BAL
2,7,2026-10-25,CLE,TEN
3,7,2026-10-25,DEN,ARI
4,7,2026-10-25,GB,DET
5,7,2026-10-25,IND,MIN
6,7,2026-10-25,KC,SEA
7,7,2026-10-25,LA,LV
8,7,2026-10-25,MIA,NYJ
9,7,2026-10-25,NYG,HOU


In [11]:

completed = completed.loc[
    (completed["week"] < TARGET_WEEK)
    & completed["home_score"].notna()
    & completed["away_score"].notna()
].copy()

home_records = completed[
    ["home_team", "home_score", "away_score"]
].rename(columns={
    "home_team": "team",
    "home_score": "points_for",
    "away_score": "points_against"
})

away_records = completed[
    ["away_team", "away_score", "home_score"]
].rename(columns={
    "away_team": "team",
    "away_score": "points_for",
    "home_score": "points_against"
})

team_results = pd.concat(
    [home_records, away_records],
    ignore_index=True
)

team_results["wins"] = (
    team_results["points_for"] > team_results["points_against"]
).astype(int)

team_results["losses"] = (
    team_results["points_for"] < team_results["points_against"]
).astype(int)

team_results["ties"] = (
    team_results["points_for"] == team_results["points_against"]
).astype(int)

records = team_results.groupby("team", as_index=False).agg(
    wins=("wins", "sum"),
    losses=("losses", "sum"),
    ties=("ties", "sum"),
    games_played=("wins", "size"),
    points_for=("points_for", "sum"),
    points_against=("points_against", "sum")
)

records["point_differential"] = (
    records["points_for"] - records["points_against"]
)

display(records.sort_values(
    ["wins", "point_differential"],
    ascending=[False, False]
))


,team,wins,losses,ties,games_played,points_for,points_against,point_differential
28,SF,6,0,0,6,177.0,110.0,67.0
15,KC,5,0,0,5,140.0,83.0,57.0
20,MIN,5,0,0,5,109.0,64.0,45.0
3,BUF,5,1,0,6,197.0,171.0,26.0
14,JAX,4,2,0,6,154.0,104.0,50.0
5,CHI,4,2,0,6,150.0,119.0,31.0
2,BAL,4,2,0,6,156.0,132.0,24.0
21,NE,4,2,0,6,136.0,113.0,23.0
26,PIT,4,2,0,6,137.0,128.0,9.0
11,GB,4,2,0,6,127.0,125.0,2.0


In [12]:

required_strength_cols = [
    "team",
    "weekly_team_strength"
]

required_injury_cols = [
    "team",
    "injury_adjustment",
    "injury_adjusted_team_strength"
]

for name, frame, columns in [
    ("strength", strength, required_strength_cols),
    ("injury", injury_strength, required_injury_cols)
]:
    missing = set(columns) - set(frame.columns)
    if missing:
        raise KeyError(
            f"{name} missing columns: {sorted(missing)}"
        )

team_snapshot = strength[
    required_strength_cols
].merge(
    injury_strength[required_injury_cols],
    on="team",
    how="left",
    validate="one_to_one"
).merge(
    records,
    on="team",
    how="left",
    validate="one_to_one"
)

assert len(team_snapshot) == 32
assert team_snapshot["team"].is_unique
assert team_snapshot["weekly_team_strength"].notna().all()

print("Teams:", len(team_snapshot))
display(team_snapshot.head(10))


Teams: 32


,team,weekly_team_strength,injury_adjustment,injury_adjusted_team_strength,wins,losses,ties,games_played,points_for,points_against,point_differential
0,SF,5.084675,-0.792600,4.292075,6,0,0,6,177.0,110.0,67.0
1,BUF,4.947555,0.000000,4.947555,5,1,0,6,197.0,171.0,26.0
2,KC,4.051178,-0.250875,3.800303,5,0,0,5,140.0,83.0,57.0
3,SEA,3.904552,-0.490575,3.413977,3,3,0,6,148.0,122.0,26.0
4,NE,3.615375,-0.849525,2.765850,4,2,0,6,136.0,113.0,23.0
5,DEN,3.197866,-0.297300,2.900566,4,2,0,6,114.0,125.0,-11.0
6,MIN,3.140947,-0.386925,2.754022,5,0,0,5,109.0,64.0,45.0
7,JAX,3.094193,0.000000,3.094193,4,2,0,6,154.0,104.0,50.0
8,DET,2.858510,-1.363725,1.494785,3,2,0,5,164.0,148.0,16.0
9,LA,2.769153,-0.503700,2.265453,2,4,0,6,135.0,138.0,-3.0


In [13]:

team_snapshot["season"] = SEASON
team_snapshot["snapshot_week"] = TARGET_WEEK

remaining_games["season"] = SEASON
remaining_games["snapshot_week"] = TARGET_WEEK

team_snapshot.to_parquet(
    OUTPUT_DIR / "team_snapshot.parquet",
    index=False
)

remaining_games.to_parquet(
    OUTPUT_DIR / "remaining_schedule.parquet",
    index=False
)

team_snapshot.to_csv(
    OUTPUT_DIR / "team_snapshot.csv",
    index=False
)

remaining_games[
    ["week", "gameday", "away_team", "home_team"]
].to_csv(
    OUTPUT_DIR / "remaining_schedule.csv",
    index=False
)

print("Saved Week 7 ROS inputs.")


Saved Week 7 ROS inputs.
